# Mini-projet : recommander une recette

| Module | Durée | Prérequis |
|---|---|---|
| M1 – Vecteurs et ressemblance | 150 min | M0 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Un site de cuisine affiche, sous chaque recette, un encadré « **Vous aimerez aussi** » avec trois autres recettes. Aujourd'hui, ces suggestions sont choisies à la main. Le site voudrait les calculer automatiquement à partir des ingrédients.

Vous disposez d'un fichier de 19 recettes (`donnees/recettes.csv`) qui donne, pour une portion, la quantité en grammes de 17 ingrédients.

**Activité** : Quelles informations seraient utiles pour faire de bonnes suggestions, mais manquent dans ce fichier ? (Pensez au temps de préparation, au type de plat, aux avis des lecteurs…) Lesquelles peut-on raisonnablement ignorer pour une première version ?

**Activité** : Formulez la question précise à laquelle le calcul répond.

**Activité** : Écrivez vos hypothèses. En particulier : « deux recettes se ressemblent si elles utilisent les mêmes ingrédients » — en mêmes quantités, ou seulement les mêmes ingrédients ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_suggestions_crepes = ["gaufres", "quatre-quarts", "flan"]   # vos 3 suggestions sous « crêpes » (modifiez)

## ③ Abstraire

Chaque recette devient un **vecteur de dimension 17** : une coordonnée par ingrédient. Recommander, c'est trouver les 3 vecteurs les plus « proches » de celui d'une recette donnée.

Vous connaissez trois mesures depuis le notebook 1.3. Mais il y a un nouveau choix à faire, sur les **données elles-mêmes** :

| Représentation | Exemple pour les crêpes | Ce qui compte |
|---|---|---|
| **Quantités** en grammes | (60, 50, 125, 10, 10, 0, …, 1) | les ingrédients **et** leurs proportions |
| **Présence** (1 si l'ingrédient est utilisé, 0 sinon) | (1, 1, 1, 1, 1, 0, …, 1) | seulement la liste des ingrédients |

**Activité** : Avec les quantités en grammes, quel ingrédient pèse le plus dans le calcul d'une distance : 250 g de pommes de terre ou 1 g de sel ? Est-ce souhaitable ?

Indiquez vos réponses et justifiez-les.
## ④ Calculer

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("donnees/recettes.csv", index_col="recette")
recettes = list(df.index)
ingredients = list(df.columns)
Q = df.to_numpy(dtype=float)            # quantités, forme (19, 17)
B = (Q > 0).astype(float)               # présence, forme (19, 17)
print(Q.shape, B.shape)
df.head()

In [ ]:
def cosinus(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    return 0.0 if nu == 0 or nv == 0 else float(u @ v / (nu * nv))

def euclidienne(u, v):
    return float(np.linalg.norm(u - v))

def recommander(nom, X, mesure, k=3):
    # Les k recettes les plus semblables à `nom` selon `mesure` appliquée aux lignes de X
    i = recettes.index(nom)
    plus_grand_est_mieux = mesure is cosinus
    scores = [(mesure(X[i], X[j]), recettes[j]) for j in range(len(X)) if j != i]
    scores.sort(reverse=plus_grand_est_mieux)
    return [r for _, r in scores[:k]]

for recette in ["crêpes", "mousse au chocolat", "pâtes carbonara"]:
    print(f"\n{recette}")
    print("  cosinus, quantités    :", recommander(recette, Q, cosinus))
    print("  cosinus, présence     :", recommander(recette, B, cosinus))
    print("  euclidienne, quantités:", recommander(recette, Q, euclidienne))

### Corroborer : les mesures sont-elles d'accord ?

Pour chaque recette, on compte combien des 3 suggestions sont communes à deux méthodes. Un fort désaccord signale que le choix de la méthode n'est pas anodin.

In [ ]:
methodes = {
    "cos. quantités": (Q, cosinus),
    "cos. présence": (B, cosinus),
    "eucl. quantités": (Q, euclidienne),
}
sugg = {m: {r: set(recommander(r, X, f)) for r in recettes} for m, (X, f) in methodes.items()}
noms_m = list(methodes)
accord = np.array([[np.mean([len(sugg[a][r] & sugg[b][r]) for r in recettes]) for b in noms_m] for a in noms_m])
print("Nombre moyen de suggestions communes (sur 3) :")
print(pd.DataFrame(accord.round(2), index=noms_m, columns=noms_m))

### Visualiser toutes les ressemblances

In [ ]:
S = np.array([[cosinus(Q[i], Q[j]) for j in range(len(Q))] for i in range(len(Q))])
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(S, cmap="liouai")
ax.set_xticks(range(len(recettes)), recettes, rotation=90, fontsize=8)
ax.set_yticks(range(len(recettes)), recettes, fontsize=8)
fig.colorbar(im, label="similarité cosinus (quantités)")
ax.set_title("Ressemblances entre recettes")
plt.tight_layout(); plt.show()

## ⑤ Interpréter

In [ ]:
print("Vos suggestions pour les crêpes :", pari_suggestions_crepes)
for m, (X, f) in methodes.items():
    communes = set(pari_suggestions_crepes) & set(recommander("crêpes", X, f))
    print(f"  {m:16s}: {recommander('crêpes', X, f)}  — en commun avec vous : {len(communes)}")

**Activité** :

1. Pour les crêpes, quelle méthode donne les suggestions les plus convaincantes pour un lecteur du site ? Justifiez avec un exemple précis.
2. Trouvez une recette pour laquelle la distance euclidienne donne une suggestion absurde. Expliquez pourquoi, en regardant les quantités.
3. Sur la carte des ressemblances, repérez des « familles » de recettes. Correspondent-elles à des catégories qu'un cuisinier reconnaîtrait ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Hypothèse la plus fragile.** Classez ces hypothèses de la plus fragile à la plus solide, et pour la plus fragile, montrez son effet sur une recommandation :
    - tous les ingrédients ont la même importance ;
    - la quantité compte autant que la présence ;
    - deux recettes aux ingrédients semblables plaisent aux mêmes lecteurs.
- **Une piste d'amélioration.** Le sel est présent dans presque toutes les recettes salées ; il ne distingue donc pas grand-chose. Proposez une façon de donner moins de poids aux ingrédients très courants, puis testez-la.
- **Généraliser.** Écrivez une fonction `vous_aimerez_aussi(nom, representation="presence", mesure="cosinus", k=3)` qui permette de changer chaque choix par un paramètre. Testez-la en ajoutant une recette au fichier.

In [ ]:
def vous_aimerez_aussi(nom, representation="presence", mesure="cosinus", k=3):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def vous_aimerez_aussi(nom, representation="presence", mesure="cosinus", k=3, poids=None):
    X = {"presence": B, "quantites": Q}[representation]
    if poids is not None:
        X = X * poids
    f = {"cosinus": cosinus, "euclidienne": euclidienne}[mesure]
    return recommander(nom, X, f, k)

# Donner moins de poids aux ingrédients fréquents (idée proche de l'« IDF » utilisée pour les textes)
frequence = B.mean(axis=0)
poids_idf = np.log(1 / frequence)
print({i: round(float(p), 2) for i, p in zip(ingredients, poids_idf)})
for r in ["crêpes", "risotto au parmesan", "tortilla espagnole"]:
    print(r, "->", vous_aimerez_aussi(r), "| pondéré :", vous_aimerez_aussi(r, poids=poids_idf))

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Filtrer l'information pertinente": 1,
    "Identifier, énoncer et justifier les hypothèses": 1,
    "Quantifier la validité et l'impact des hypothèses": 1,
    "Corroborer ses résultats": 1,
    "Énoncer le résultat dans le contexte du problème": 1,
    "Transformer des constantes en paramètres": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "M1-projet-recettes", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")